# NeuroMAS Walkthrough

So far you have built agents by writing prompts and tools. This notebook takes the next step: **agents that learn**.

NeuroMAS connects role-free LLM agents into layers, like neurons in a network, and trains all of them from one reward: whether the final answer was correct. This walkthrough uses the project's own code, so everything you see here is what runs during training.

**You will see:**

1. What an agent is told, and what it is not told
2. How messages are routed privately between layers
3. How one reward trains every agent
4. How to run training
5. How a trained network grows

Paper: [arXiv:2605.16757](https://arxiv.org/abs/2605.16757)

## Setup

Install the dependencies, then import the project's modules. Sections 1 and 2 run without a GPU.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import engine
from tasks import TASKS

task = TASKS["arc"]

## 1. What an agent is told

A typical multi-agent prompt starts with "You are a planner." A NeuroMAS node gets only its position in the network and the output format. Look at the first node of NeuroMAS-5, which has the topology `[2, 2]`:

In [ ]:
question = "Which gas do plants absorb from the air?\nA. Oxygen\nB. Carbon dioxide\nC. Nitrogen\nD. Helium"
print(engine.node_prompt(task, [2, 2], layer=0, node=0, q=question, incoming=[]))

There is no role in this prompt. If this node becomes good at, say, eliminating wrong options, that behavior came from training.

The last layer receives private messages and is asked for an answer:

In [ ]:
incoming = [(0, 0, "Plants take in CO2 for photosynthesis."), (0, 1, "Option B looks right.")]
print(engine.node_prompt(task, [2, 2], layer=1, node=0, q=question, incoming=incoming))

## 2. How messages are routed

An upstream node writes one block per recipient. `parse_sections` turns that text into private messages, and each downstream node reads only its own. This parser is the wiring of the network.

In [ ]:
output = "TO #1: Focus on what photosynthesis consumes.\nTO #2: Double-check option B."
sections, followed_format = engine.parse_sections(output, num_recipients=2)
for recipient, message in enumerate(sections, start=1):
    print(f"Node {recipient} receives: {message}")
print("Followed the format:", followed_format)

If a node ignores the format, every recipient gets its full output, so a forward pass never breaks:

In [ ]:
sections, followed_format = engine.parse_sections("I think the answer is B.", num_recipients=2)
print(sections, followed_format)

## 3. How one reward trains every agent

After a forward pass, `train_stage` scores the final answer once:

```python
r = 1.0 if task.score(st, ex) else 0.0
```

Then each node takes its own gradient step with that same `r`:

```python
baselines[nidx] = config.EMA_DECAY * baselines[nidx] + (1 - config.EMA_DECAY) * r
loss = -(lp.mean() * (r - baselines[nidx]))
```

- `lp` holds the log-probabilities of the tokens this node generated.
- `baselines[nidx]` is this node's running average reward.
- `r - baseline` is positive when the team did better than usual, so the text this node wrote becomes more likely. When the team did worse than usual, it becomes less likely.

No one tells a node whether its own message helped. Over many queries, messages that tend to lead to correct answers are reinforced.

All agents live on one backbone. `AM` keeps each node's LoRA adapter and swaps the active one in place:

```python
am.sw(nidx)  # make node nidx the active agent
```

## 4. Run training

This smoke test uses tiny budgets to confirm the whole pipeline works on your machine. It needs a CUDA GPU.

In [ ]:
!SMOKE=1 python run.py --task navigate --size 3

For a real run, use the full budget from `config.py`:

```bash
python run.py --task arc --size 3
```

## 5. Grow the network

Requesting `--size 7` trains NeuroMAS-3, grows it to NeuroMAS-5, then to NeuroMAS-7, and reports every stage. These mappings show where each trained node moves in the larger network:

In [ ]:
print("NeuroMAS-3 to -5 (old index -> new index):", engine.grow_2_to_4())
print("NeuroMAS-5 to -7 (old index -> new index):", engine.grow_4_to_6())

## Where to go next

- Change `Task.score` in `tasks.py` to train the team on your own objective.
- Add a benchmark with a loader and one entry in `TASKS`.
- Print the outputs of `engine.fwd` before and after training and compare what the agents say to each other.
- Read the [paper](https://arxiv.org/abs/2605.16757) for the theory and the full experiments.